In [1]:
import os

In [2]:
%pwd

'e:\\Projects\\Text-Summarization\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\Projects\\Text-Summarization'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            eval_strategy=params.eval_strategy,
            eval_steps=params.eval_steps,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps
        )

        return model_trainer_config


In [8]:
import os
import torch
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    AutoModelForSeq2SeqLM,
    AutoTokenizer
)
from datasets import load_from_disk, Dataset, load_dataset
from textSummarizer.logging import logger

c:\Users\admin\anaconda3\envs\summary\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info(f"Using device: {device.upper()}")

        model_ckpt = "t5-small"  
        tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)
        logger.info("Model and tokenizer loaded successfully")

        logger.info("Loading small CNN/DailyMail dataset subset...")
        dataset = load_dataset("abisee/cnn_dailymail", "1.0.0")

        dataset["train"] = dataset["train"].select(range(300))
        dataset["validation"] = dataset["validation"].select(range(60))

        def preprocess_function(examples):
            inputs = [doc for doc in examples["article"]]
            targets = [t for t in examples["highlights"]]
            model_inputs = tokenizer(
                inputs, max_length=512, truncation=True
            )
            labels = tokenizer(
                targets, max_length=128, truncation=True
            )
            model_inputs["labels"] = labels["input_ids"]
            return model_inputs

        tokenized_datasets = dataset.map(
            preprocess_function, batched=True, remove_columns=["article", "highlights", "id"]
        )

        data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=1,
            per_device_train_batch_size=2,
            per_device_eval_batch_size=2,
            warmup_steps=10,
            weight_decay=0.01,
            logging_steps=5,
            eval_strategy="steps",
            eval_steps=50,
            save_steps=50,
            remove_unused_columns=False,  
            logging_dir=os.path.join(self.config.root_dir, "logs"),
            load_best_model_at_end=True
        )

        trainer = Trainer(
            model=model,
            args=trainer_args,
            tokenizer=tokenizer,
            data_collator=data_collator,
            train_dataset=tokenized_datasets["train"],
            eval_dataset=tokenized_datasets["validation"]
        )

        logger.info("Starting model training...")
        trainer.train()
        logger.info("Model training completed successfully")

        model_dir = os.path.join(self.config.root_dir, "t5-small-cnn-model")
        tokenizer_dir = os.path.join(self.config.root_dir, "tokenizer")

        model.save_pretrained(model_dir)
        tokenizer.save_pretrained(tokenizer_dir)
        logger.info(f"Model saved at: {model_dir}")
        logger.info(f"Tokenizer saved at: {tokenizer_dir}")

In [12]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e


YAML file loaded successfully: config\config.yaml
[2025-11-03 15:48:43,854: INFO: common: YAML file loaded successfully: config\config.yaml]
YAML file loaded successfully: params.yaml
[2025-11-03 15:48:43,870: INFO: common: YAML file loaded successfully: params.yaml]
Created directory at: artifacts
[2025-11-03 15:48:43,875: INFO: common: Created directory at: artifacts]
Created directory at: artifacts/model_trainer
[2025-11-03 15:48:43,882: INFO: common: Created directory at: artifacts/model_trainer]
Using device: CPU
[2025-11-03 15:48:43,899: INFO: 434286381: Using device: CPU]
Model and tokenizer loaded successfully
[2025-11-03 15:48:46,343: INFO: 434286381: Model and tokenizer loaded successfully]
Loading small CNN/DailyMail dataset subset...
[2025-11-03 15:48:46,347: INFO: 434286381: Loading small CNN/DailyMail dataset subset...]


Map: 100%|██████████| 11490/11490 [00:39<00:00, 287.75 examples/s]
C:\Users\admin\AppData\Local\Temp\ipykernel_5688\434286381.py:61: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting model training...
[2025-11-03 15:49:33,559: INFO: 434286381: Starting model training...]


c:\Users\admin\anaconda3\envs\summary\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
50,2.466800,2.075074
100,1.960400,2.038049
150,2.467300,2.034367


c:\Users\admin\anaconda3\envs\summary\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\admin\anaconda3\envs\summary\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Model training completed successfully
[2025-11-03 16:05:52,748: INFO: 434286381: Model training completed successfully]
Model saved at: artifacts/model_trainer\t5-small-cnn-model
[2025-11-03 16:05:54,186: INFO: 434286381: Model saved at: artifacts/model_trainer\t5-small-cnn-model]
Tokenizer saved at: artifacts/model_trainer\tokenizer
[2025-11-03 16:05:54,195: INFO: 434286381: Tokenizer saved at: artifacts/model_trainer\tokenizer]
